# Notebook 2: LSTM-SNP with Fuzzy Feature Augmentation

**Dataset**: Dow Jones Industrial Index

## Description
This notebook implements **fuzzy feature augmentation** for the LSTM-SNP model. A Takagi-Sugeno 
fuzzy inference system processes the current and previous input values to generate an additional 
feature, which is concatenated with the original input before being fed into the unmodified 
LSTM-SNP cell.

The LSTM-SNP cell itself is **NOT modified** — only the input representation is enriched with 
fuzzy-derived features.

## Theory: Fuzzy Feature Augmentation

### LSTM-SNP Cell (Unchanged)
The LSTM-SNP cell equations remain exactly as in the baseline.

### Fuzzy Inference System
A Takagi-Sugeno fuzzy system augments the input:

**Membership Functions** (Fixed Gaussian):
- $\mu_{low}(x) = \exp\left(-\frac{(x - (-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(x) = \exp\left(-\frac{(x - (+1))^2}{2 \cdot 0.5^2}\right)$

**Rules** (4 Takagi-Sugeno rules):
1. IF $x(t)$ is low AND $x(t-1)$ is low → $y_1 = a_1 x(t) + b_1 x(t-1) + c_1$
2. IF $x(t)$ is low AND $x(t-1)$ is high → $y_2 = a_2 x(t) + b_2 x(t-1) + c_2$
3. IF $x(t)$ is high AND $x(t-1)$ is low → $y_3 = a_3 x(t) + b_3 x(t-1) + c_3$
4. IF $x(t)$ is high AND $x(t-1)$ is high → $y_4 = a_4 x(t) + b_4 x(t-1) + c_4$

**Defuzzification**: $y_{fuzzy} = \frac{\sum_i w_i y_i}{\sum_i w_i}$ where $w_i = \prod_j \mu_j(x_j)$

**Augmented Input**: $x'(t) = [x(t), y_{fuzzy}(t)]$

## Model Architecture & Implementation

In [ ]:
# ============================================================
# ALL IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

### Fuzzy Inference System (NumPy — for preprocessing)

In [ ]:
# ============================================================
# Fuzzy Inference System (NumPy — for preprocessing)
#
# Fixed Gaussian membership functions:
#   μ_low(x)  = exp(-(x - (-1))² / (2·0.5²))
#   μ_high(x) = exp(-(x - (+1))² / (2·0.5²))
#
# 4 Takagi-Sugeno rules with fixed consequent parameters:
#   IF x(t) is low  AND x(t-1) is low  → y₁ = 0.5·x(t) + 0.5·x(t-1)
#   IF x(t) is low  AND x(t-1) is high → y₂ = 0.7·x(t) + 0.3·x(t-1) - 0.1
#   IF x(t) is high AND x(t-1) is low  → y₃ = 0.3·x(t) + 0.7·x(t-1) + 0.1
#   IF x(t) is high AND x(t-1) is high → y₄ = 0.5·x(t) + 0.5·x(t-1)
#
# Output: y = Σ(wᵢ·yᵢ) / Σ(wᵢ)
# ============================================================

def gaussian_mf(x, center, sigma=0.5):
    """Fixed Gaussian membership function."""
    return np.exp(-(x - center)**2 / (2 * sigma**2))

def fuzzy_inference_numpy(x_t, x_tm1):
    """
    Compute fuzzy feature from x(t) and x(t-1).
    Uses fixed membership functions and fixed consequent parameters.
    """
    # Membership degrees
    mu_low_xt = gaussian_mf(x_t, center=-1.0)
    mu_high_xt = gaussian_mf(x_t, center=1.0)
    mu_low_xtm1 = gaussian_mf(x_tm1, center=-1.0)
    mu_high_xtm1 = gaussian_mf(x_tm1, center=1.0)

    # Rule firing strengths (product)
    w1 = mu_low_xt * mu_low_xtm1      # low-low
    w2 = mu_low_xt * mu_high_xtm1     # low-high
    w3 = mu_high_xt * mu_low_xtm1     # high-low
    w4 = mu_high_xt * mu_high_xtm1    # high-high

    # Consequent outputs (fixed linear functions)
    y1 = 0.5 * x_t + 0.5 * x_tm1
    y2 = 0.7 * x_t + 0.3 * x_tm1 - 0.1
    y3 = 0.3 * x_t + 0.7 * x_tm1 + 0.1
    y4 = 0.5 * x_t + 0.5 * x_tm1

    # Weighted average defuzzification
    numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
    denominator = w1 + w2 + w3 + w4 + 1e-8

    return numerator / denominator

### LSTM-SNP Cell

In [ ]:
# ============================================================
# LSTM-SNP Cell (Original — Unmodified)
# ============================================================

class LSTMSNPCell(layers.Layer):
    """
    LSTM-SNP Cell: A long short-term memory model inspired from
    spiking neural P systems.

    Gates:
      r(t) = ρ(W_r x(t) + U_r u(t-1) + b_r)   [reset]
      c(t) = ρ(W_c x(t) + U_c u(t-1) + b_c)   [consumption]
      o(t) = ρ(W_o x(t) + U_o u(t-1) + b_o)   [output/generation]
      a(t) = f(W_a x(t) + U_a u(t-1) + b_a)   [generated spikes]

    State update:
      u(t) = r(t) * u(t-1) - c(t) * a(t)
      h(t) = o(t) * a(t)

    ρ = hard_sigmoid, f = tanh
    """
    def __init__(self, units,
                 activation='tanh',
                 recurrent_activation='hard_sigmoid',
                 **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = units
        self.output_size = units

        self.activation = tf.keras.activations.get(activation)
        self.recurrent_activation = tf.keras.activations.get(recurrent_activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.kernel = self.add_weight(
            shape=(input_dim, self.units * 4),
            initializer='glorot_uniform',
            name='kernel'
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, self.units * 4),
            initializer='orthogonal',
            name='recurrent_kernel'
        )
        self.bias = self.add_weight(
            shape=(self.units * 4,),
            initializer='zeros',
            name='bias'
        )

    def call(self, inputs, states):
        u_tm1 = states[0]

        z = tf.matmul(inputs, self.kernel) + \
            tf.matmul(u_tm1, self.recurrent_kernel) + self.bias

        z0 = z[:, :self.units]
        z1 = z[:, self.units:2*self.units]
        z2 = z[:, 2*self.units:3*self.units]
        z3 = z[:, 3*self.units:]

        r = self.recurrent_activation(z0)  # reset
        c = self.recurrent_activation(z1)  # consumption
        o = self.recurrent_activation(z2)  # output/generation
        a = self.activation(z3)            # generated spikes

        u = r * u_tm1 - c * a  # internal state
        h = o * a              # output

        return h, [u]

    def get_config(self):
        config = super().get_config()
        config.update({
            'units': self.units,
        })
        return config

### Build Model

In [ ]:
# ============================================================
# Model Construction: LSTM-SNP with Fuzzy Feature Augmentation
# input_dim=2: [x(t), y_fuzzy(t)]
# The LSTM-SNP cell is UNMODIFIED.
# ============================================================

def build_model(input_dim, units, batch_size):
    cell = LSTMSNPCell(units)
    rnn = layers.RNN(cell, return_sequences=False, stateful=True)

    inputs = tf.keras.Input(batch_shape=(batch_size, 1, input_dim))
    x = rnn(inputs)
    outputs = layers.Dense(1)(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [ ]:
# Quick model check
model = build_model(input_dim=2, units=8, batch_size=1)
model.summary()

## Data Pipeline — Dow Jones Industrial Index

In [ ]:
import os
for f in os.listdir('/kaggle/input/datasets/satabartosarkar123/monthly-closings-of-the-dowjones-csv'):
    print(f)

In [ ]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    '/kaggle/input/datasets/satabartosarkar123/monthly-closings-of-the-dowjones-csv/monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)

original_raw_values = series.values.flatten()
print(f"Base data shape: {original_raw_values.shape}")
print(f"First 5 values: {original_raw_values[:5]}")


In [ ]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

In [ ]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

In [ ]:
# ============================================================
# 4-5. Train-Test Split + Scaling (inside noise loop)
# 6. Reshape with Fuzzy Feature Augmentation (inside noise loop)
# ============================================================
print("Split: last 60 for test, rest for training")
print("Fuzzy augmentation: fuzzy_inference_numpy(x_t, x_tm1) → input_dim=2")
print("Lag=1, sequential x_tm1 from previous sample")

In [ ]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================
print("Reshape: (samples, 1, 2) — original + fuzzy feature")

## Training Loop

In [ ]:
# ============================================================
# Gaussian Noise Evaluation + 60-Run Experiment Protocol (Keras)
# ============================================================
s_x = np.std(original_raw_values)

noise_levels = [0.0, 0.005, 0.05, 0.10]
_molab_results = []

for lam in noise_levels:
    sigma = lam * s_x
    print("\n" + "="*80)
    print(f"EVALUATING NOISE LEVEL: {lam*100:.1f}% (lambda={lam}, sigma={sigma:.6f})")
    print("="*80 + "\n")
    
    np.random.seed(42)
    tf.random.set_seed(42)
    noise = np.random.normal(0, sigma, size=original_raw_values.shape)
    raw_values = original_raw_values + noise
    
    # 2. Difference
    diff_values = difference(raw_values, 1)
    # 3. Supervised (lag=1)
    supervised = timeseries_to_supervised(diff_values, 1)
    # 4. Split (Last 60 for test, no validation)
    train, test = supervised[:-60], supervised[-60:]
    # 5. Scale
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train)
    train_scaled = scaler.transform(train)
    test_scaled = scaler.transform(test)
    
    # 6. Reshape with Fuzzy Feature Augmentation (input_dim=2)
    #    x_t = current scaled input, x_tm1 = previous sample's scaled input
    X_train_raw = train_scaled[:, 0:-1]  # shape (N, 1)
    y_train = train_scaled[:, -1]
    
    X_train_fuzzy = np.zeros((X_train_raw.shape[0], 2))
    for i in range(X_train_raw.shape[0]):
        x_t = X_train_raw[i, 0]
        x_tm1 = X_train_raw[i-1, 0] if i > 0 else 0.0
        y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
        X_train_fuzzy[i, 0] = x_t
        X_train_fuzzy[i, 1] = y_fuzz
    
    X_train = X_train_fuzzy.reshape((X_train_fuzzy.shape[0], 1, 2))
    
    X_test_raw = test_scaled[:, 0:-1]
    y_test = test_scaled[:, -1]
    
    X_test_fuzzy = np.zeros((X_test_raw.shape[0], 2))
    for i in range(X_test_raw.shape[0]):
        x_t = X_test_raw[i, 0]
        x_tm1 = X_test_raw[i-1, 0] if i > 0 else 0.0
        y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
        X_test_fuzzy[i, 0] = x_t
        X_test_fuzzy[i, 1] = y_fuzz

    X_test = X_test_fuzzy.reshape((X_test_fuzzy.shape[0], 1, 2))
    
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    
    # Metrics go to disk to save RAM; only best run stays in memory
    import tempfile, csv
    metrics_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, prefix=f'metrics_noise{lam}_')
    metrics_writer = csv.writer(metrics_file)
    metrics_writer.writerow(['run', 'rmse', 'mse', 'nmse'])
    best_rmse = float('inf')
    best_mse = None
    best_nmse = None
    best_predictions = None
    best_losses = None
    
    # 60 RUNS, 100 EPOCHS
    for run in range(60):
        print(f'\n===== RUN {run+1}/60 [Noise {lam*100:.1f}%] =====')

        np.random.seed(run)
        tf.random.set_seed(run)

        import gc
        gc.collect()
        tf.keras.backend.clear_session()
        model = build_model(input_dim=2, units=8, batch_size=1)

        rnn_layer = model.layers[1]
        cell = rnn_layer.cell
        weights = cell.get_weights()
        bias = weights[2].copy()
        bias[8:16] = 1.0
        weights[2] = bias
        cell.set_weights(weights)

        run_losses = []
        for epoch in range(100):
            history = model.fit(
                X_train, y_train,
                epochs=1, batch_size=1,
                verbose=0, shuffle=False
            )
            run_losses.append(history.history['loss'][0])
            if (epoch+1) % 10 == 0:
                print(f'  Epoch {epoch+1}/100, Loss: {history.history["loss"][0]:.6f}')
            rnn_layer.reset_states()


        print(f'Training complete for run {run+1}')

        # Warm-up: condition hidden states on training data
        model.predict(X_train, batch_size=1, verbose=0)

        # Test predictions (single-step)
        predictions = []
        for i in range(len(test_scaled)):
            X = test_scaled[i, 0:-1]
            x_t = X[0]
            x_tm1 = test_scaled[i-1, 0] if i > 0 else train_scaled[-1, 0]
            y_fuzz = fuzzy_inference_numpy(x_t, x_tm1)
            X_aug = np.array([x_t, y_fuzz]).reshape(1, 1, 2)
            yhat = model.predict(X_aug, batch_size=1, verbose=0)[0, 0]

            # Invert scaling
            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]

            # Invert differencing
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)
            expected = raw_values[len(train) + i + 1]

        # Compute metrics
        actual = raw_values[-len(predictions):]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
        nmse = mse / np.power(dominator, 2)
        # Write metrics to disk (not RAM)
        metrics_writer.writerow([run+1, rmse, mse, nmse])
        metrics_file.flush()
        if rmse < best_rmse:
            best_rmse = rmse
            best_mse = mse
            best_nmse = nmse
            best_predictions = predictions[:]
            best_losses = run_losses[:]
            print(f'Run {run+1} — RMSE: {rmse:.6f} *** NEW BEST ***')
        else:
            print(f'Run {run+1} — RMSE: {rmse:.6f}')


    # Read all 60 runs' metrics back from disk for statistics
    metrics_file.close()
    import csv as csv_reader
    with open(metrics_file.name, 'r') as mf:
        reader = csv_reader.DictReader(mf)
        disk_rmse, disk_mse, disk_nmse = [], [], []
        for row in reader:
            disk_rmse.append(float(row['rmse']))
            disk_mse.append(float(row['mse']))
            disk_nmse.append(float(row['nmse']))
    os.remove(metrics_file.name)  # cleanup temp file
    
    print(f'\n===== STATS FOR NOISE {lam*100:.1f}% (60 runs) =====')
    print(f'RMSE: {np.mean(disk_rmse):.6f} +/- {np.std(disk_rmse):.6f}')
    print(f'MSE:  {np.mean(disk_mse):.6f} +/- {np.std(disk_mse):.6f}')
    print(f'NMSE: {np.mean(disk_nmse):.10f} +/- {np.std(disk_nmse):.10f}')
    
    _molab_results.append([lam, 
        np.mean(disk_rmse), np.std(disk_rmse),
        np.mean(disk_mse), np.std(disk_mse),
        np.mean(disk_nmse), np.std(disk_nmse),
        best_rmse, best_predictions, best_losses, raw_values.copy()])
    print(f"\nBest for {lam*100:.1f}% noise: RMSE={best_rmse:.6f}")


## Results

In [ ]:
# ============================================================
# Final Metrics Summary & Plots per Noise Level
# ============================================================
from tabulate import tabulate

for entry in _molab_results:
    lam = entry[0]
    mean_rmse, std_rmse = entry[1], entry[2]
    mean_mse, std_mse = entry[3], entry[4]
    mean_nmse, std_nmse = entry[5], entry[6]
    best_rmse_val = entry[7]
    best_predictions = entry[8]
    best_losses = entry[9]
    raw_values_for_plot = entry[10]
    
    print("\n" + "="*80)
    print(f"RESULTS FOR NOISE LEVEL: {lam*100:.1f}%")
    print("="*80)
    
    print(f'Mean RMSE: {mean_rmse:.6f} +/- {std_rmse:.6f}')
    print(f'Mean MSE:  {mean_mse:.6f} +/- {std_mse:.6f}')
    print(f'Mean NMSE: {mean_nmse:.10f} +/- {std_nmse:.10f}')
    print(f'Best RMSE: {best_rmse_val:.6f}')
    
    actual = raw_values_for_plot[-60:]
    
    plt.figure(figsize=(12, 5))
    plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
    plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
             linewidth=1.5, linestyle='--')
    plt.title(f'Predictions vs Actual (Noise {lam*100:.1f}%)')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(best_losses, color='green', linewidth=1.0)
    plt.title(f'Training Loss (Best Run, Noise {lam*100:.1f}%)')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("FINAL RESULTS TABLE ACROSS ALL NOISE LEVELS")
print("="*80)
df_res = pd.DataFrame(
    [[r[0], f'{r[1]:.6f} +/- {r[2]:.6f}', f'{r[3]:.6f} +/- {r[4]:.6f}', 
      f'{r[5]:.10f} +/- {r[6]:.10f}', f'{r[7]:.6f}'] for r in _molab_results],
    columns=['Noise Level', 'RMSE (mean+/-std)', 'MSE (mean+/-std)', 'NMSE (mean+/-std)', 'Best RMSE']
)
print(tabulate(df_res, headers='keys', tablefmt='github', showindex=False))


## Observations

### Gaussian Noise robustness on Dow Jones Industrial Index

**Run the notebook to populate results.**